### Fine-tuned model evaluation

The purpose of this last notebook is to evaluate the fine-tuned `flan-t5-base` model across 4 quantization levels: FP32, BF16, INT8 and INT4.

The evaluation principles used here are the same as for the raw models:
* same main evaluation metrics: Exact Match & F1;
* same test split, tokenizer, quantization settings;
* same evaluation function `evaluate_model()` (inside of src/evaluate.py).

In [1]:
# github repo pull & dependencies installation for the evaluation
!git clone https://github.com/Julianlls/DeepLearning.git
%cd DeepLearning
!pip install -q transformers datasets accelerate bitsandbytes

Cloning into 'DeepLearning'...
remote: Enumerating objects: 129, done.
remote: Counting objects: 100% (129/129), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 129 (delta 62), reused 91 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (129/129), 443.53 KiB | 1.68 MiB/s, done.
Resolving deltas: 100% (62/62), done.
/content/DeepLearning
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 18.1 MB/s eta 0:00:00


To follow on the above-set conditions, in terms of the set-up we will :
*    import the src directory to reuse load_processed() and evaluate_model();
*    load the test split (3000 questions) using load_processed();
*    set the target hardware, using a GPU for INT8 and INT4 evaluation;
*    load the google/flan-t5-base tokenizer to maintain identical text tokenization.


In [2]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd()))

from src.evaluate import evaluate_model
from src.preprocessing import load_processed
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, BitsAndBytesConfig
import torch

ds = load_processed()
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"device: {device}")

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/27000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3000 [00:00<?, ? examples/s]

device: cuda


We also need to define how the fine-tuned model loads for each precision level we evaluate:

*   FP32 - standard loading using 32 bits per weight.
    *    our reference baseline;
*   BF16 - 16 bits per weight
    *    reduces the memory usage by half compared to FP32;
*   INT8 and INT4: 8 or 4 bits per weight using the bitsandbytes library.
    *    INT8 and INT4 models automatically load directly onto the GPU;

  
`load_model()` here loads the model at the specified quantization and transfers it to the GPU. It raises an error for unsupported quantization names.


In [3]:
FT_MODEL_ID = "Smambu/flan-t5-base-adversarialqa-ft"

KWARGS = {
    "fp32": {},
    "bf16": {"torch_dtype": torch.bfloat16},
    "int8": {"quantization_config": BitsAndBytesConfig(load_in_8bit = True), "device_map": "auto"},
    "int4": {"quantization_config": BitsAndBytesConfig(load_in_4bit = True), "device_map": "auto"},
}

def load_model(model_id, precision_mode):
    if precision_mode not in KWARGS:
        raise ValueError(f"Unknown precision_mode: {precision_mode!r}")

    model = AutoModelForSeq2SeqLM.from_pretrained(
        model_id, **KWARGS[precision_mode]
    )

    if precision_mode in ("fp32", "bf16"):
        model = model.to(device)

    return model

We will skip the repeated bitsandbytes warnings during INT8/INT4 batch generation to keep the output clean without affecting the evaluation process; only genuine errors remain visible.

In [4]:
import logging
logging.getLogger("bitsandbytes.autograd._functions").setLevel(logging.ERROR)

With the set-up being completed, we will now evaluate the fine-tuned model on the test set across all quantization levels.

We re-run FP32 here, even though it was previously evaluated in `04_finetuning` (F1 = 0.550), to ensure all fine-tuned results come from the same run and code path.

In [5]:
import pandas as pd

precisions = ["fp32", "bf16", "int8", "int4"]
rows = []

for precision_mode in precisions:
    print(f"--- flan-t5-base (fine-tuned) / {precision_mode} ---")

    model = load_model(FT_MODEL_ID, precision_mode)
    row, preds = evaluate_model(
        model, tokenizer, ds["test"],
        model_name="flan-t5-base",
        state="finetuned",
        precision_mode=precision_mode,
    )
    rows.append(row)
    print("\n", row, sep="")

    del model
    if device == 'mps': torch.mps.empty_cache()
    else: torch.cuda.empty_cache()

pd.DataFrame(rows)

--- flan-t5-base (fine-tuned) / fp32 ---


config.json:   0%|          | 0.00/1.56k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generating:   0%|          | 0/375 [00:00<?, ?it/s]


{'model_name': 'flan-t5-base', 'state': 'finetuned', 'precision_mode': 'int4', 'exact_match': 0.434, 'precision': 0.56058, 'recall': 0.5618, 'f1': 0.53785, 'n_examples': 3000}


,model_name,state,precision_mode,exact_match,precision,recall,f1,n_examples
0,flan-t5-base,finetuned,fp32,0.44400,0.56737,0.58402,0.55009,3000
1,flan-t5-base,finetuned,bf16,0.44233,0.56610,0.58378,0.54905,3000
2,flan-t5-base,finetuned,int8,0.44533,0.56744,0.58562,0.55128,3000
3,flan-t5-base,finetuned,int4,0.43400,0.56058,0.56180,0.53785,3000


The FP32 evaluation passed successfully. An Exact Match (EM) score of 0.444 matches the result from `04_finetuning`, confirming that the Hub checkpoint is identical to the model evaluated earlier.

We will also output the predictions in the results/ directory.

In [ ]:
import json
from src.config import RESULTS_DIR

RESULTS_DIR.mkdir(parents = True, exist_ok = True)

precisions = ["fp32", "bf16", "int8", "int4"]
rows = []

for precision_mode in precisions:
    print(f"--- flan-t5-base (fine-tuned) / {precision_mode} ---")

    model = load_model(FT_MODEL_ID, precision_mode)
    row, preds = evaluate_model(
        model, tokenizer, ds["test"],
        model_name="flan-t5-base",
        state="finetuned",
        precision_mode=precision_mode,
    )
    rows.append(row)
    print("\n", row, sep="")

    # save predictions in the same format as the baseline files in results/
    pred_path = RESULTS_DIR / f"preds_{row['model_name']}_{row['state']}_{precision_mode}.json"
    pred_path.write_text(json.dumps(preds, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"predictions saved -> {pred_path}")

    del model
    if device == 'mps': torch.mps.empty_cache()
    else: torch.cuda.empty_cache()

pd.DataFrame(rows)

### Base vs. fine-tuned models comparison

In [10]:
columns = ["model_name", "state", "precision_mode", "exact_match", "precision", "recall", "f1"]

# raw models
raw = pd.DataFrame([
    ["flan-t5-base", "raw", "fp32", 0.42367, 0.55871, 0.55531, 0.53332],
    ["flan-t5-base", "raw", "bf16", 0.42567, 0.55996, 0.55622, 0.53420],
    ["flan-t5-base", "raw", "int8", 0.42200, 0.55709, 0.55543, 0.53232],
    ["flan-t5-base", "raw", "int4", 0.40333, 0.54037, 0.52487, 0.50944],
], columns=columns)

table = pd.concat([raw, pd.DataFrame(rows)[columns]], ignore_index=True)

# F1 change vs. each model's own FP32 row
fp32_f1 = table[table["precision_mode"] == "fp32"].set_index("state")["f1"]
table["F1 vs. FP32"] = (table["f1"] - table["state"].map(fp32_f1)).round(3)

table["Model"] = table["state"].map({"raw": "flan-t5-base", "finetuned": "flan-t5-base-finetuned"})
table["Precision"] = table["precision_mode"].str.upper()
table = table.rename(columns={"exact_match": "EM", "precision": "token_precision", "recall": "token_recall", "f1": "F1"})

table[["Model", "Precision", "EM", "token_precision", "token_recall", "F1", "F1 vs. FP32"]].round(4)

,Model,Precision,EM,token_precision,token_recall,F1,F1 vs. FP32
0,flan-t5-base,FP32,0.4237,0.5587,0.5553,0.5333,0.000
1,flan-t5-base,BF16,0.4257,0.5600,0.5562,0.5342,0.001
2,flan-t5-base,INT8,0.4220,0.5571,0.5554,0.5323,-0.001
3,flan-t5-base,INT4,0.4033,0.5404,0.5249,0.5094,-0.024
4,flan-t5-base-finetuned,FP32,0.4440,0.5674,0.5840,0.5501,0.000
5,flan-t5-base-finetuned,BF16,0.4423,0.5661,0.5838,0.5490,-0.001
6,flan-t5-base-finetuned,INT8,0.4453,0.5674,0.5856,0.5513,0.001
7,flan-t5-base-finetuned,INT4,0.4340,0.5606,0.5618,0.5379,-0.012


*    flan-t5-base = `google/flan-t5-base`
*    flan-t5-base-finetuned = `Smambu/flan-t5-base-adversarialqa-ft`

<br>

**Key evaluation comparison observations:**

*   Fine-tuning improves accuracy across all precision levels. At full precision (FP32), the fine-tuned model answers approximately 60 additional questions correctly out of 3000, increasing Exact Match accuracy from 0.424 to 0.444.
    *   Training on this dataset yields minor improvements, but overall gains are limited because the model saw the data during model's instruction tuning done by Google. We discovered this at the end of the project, so we could not redo the entire project with a different dataset.

*   BF16 and INT8 score changes remain minimal (around 0.001), representing roughly 3 questions out of 3000, which constitutes negligible noise.
    *    The weight might take 2x (BF16) or 4x (INT8) less memory without leading to strong loss in accuracy, both before and after fine-tuning.

*   INT4 is the only precision level that causes performance to drop, despite significantly reducing the model's size compared to FP32, accuracy decreases for both models.
    *    There is a limit to how much you can compress the model without impacting performance, and INT4 reaches that limit.


*   The fine-tuned model drops half as much performance at INT4. Quantizing to INT4 reduces the base model's accuracy by about 60 fewer correct answers overall (EM dropped from 0.424 to 0.403), whereas the fine-tuned model loses only about 30 (EM dropped from 0.444 to 0.434).
    *    Besides improving accuracy, fine-tuning made the model more resilient to heavy compression, making it a better choice in case strong quantization is needed.

*   The fine-tuned model answers about 90 more questions correctly than the base model at INT4, compared to a gain of 60 questions at FP32.
    *    Fine-tuning delivered its best improvement where compression was the highest.

*   A fine-tuned INT4 model does at least as well as the base FP32 model (F1 0.538 vs. 0.533 & EM 0.434 vs. 0.424).
    *    The gain from fine-tuning (F1 +0.017) is larger than the loss from INT4 compression (F1 -0.012), so aggressive quantization applied (4 bits per weight instead of 32) does not lead to a drastic loss of accuracy compared to the original full-sized model.

### All models comparison table

In [14]:
columns = ["model_name", "state", "precision_mode", "exact_match", "precision", "recall", "f1"]

# raw models
raw = pd.DataFrame([
    ["flan-t5-base",  "raw", "fp32", 0.42367, 0.55871, 0.55531, 0.53332],
    ["flan-t5-base",  "raw", "bf16", 0.42567, 0.55996, 0.55622, 0.53420],
    ["flan-t5-base",  "raw", "int8", 0.42200, 0.55709, 0.55543, 0.53232],
    ["flan-t5-base",  "raw", "int4", 0.40333, 0.54037, 0.52487, 0.50944],
    ["flan-t5-large", "raw", "fp32", 0.54833, 0.68418, 0.69684, 0.66391],
    ["flan-t5-large", "raw", "bf16", 0.54900, 0.68548, 0.69825, 0.66502],
    ["flan-t5-large", "raw", "int8", 0.54900, 0.68483, 0.69842, 0.66469],
    ["flan-t5-large", "raw", "int4", 0.53033, 0.66869, 0.68061, 0.64755],
], columns=columns)

table = pd.concat([raw, pd.DataFrame(rows)[columns]], ignore_index=True)


table["Model"] = table["model_name"].where(table["state"] == "raw", table["model_name"] + "-finetuned")

# F1 change vs. each model's own FP32 row
fp32_f1 = table[table["precision_mode"] == "fp32"].set_index("Model")["f1"]
table["F1 vs. FP32"] = (table["f1"] - table["Model"].map(fp32_f1)).round(3)

table["Precision"] = table["precision_mode"].str.upper()
table = table.rename(columns={"exact_match": "EM", "precision": "token_precision", "recall": "token_recall", "f1": "F1"})

table[["Model", "Precision", "EM", "token_precision", "token_recall", "F1", "F1 vs. FP32"]].round(4)

,Model,Precision,EM,token_precision,token_recall,F1,F1 vs. FP32
0,flan-t5-base,FP32,0.4237,0.5587,0.5553,0.5333,0.000
1,flan-t5-base,BF16,0.4257,0.5600,0.5562,0.5342,0.001
2,flan-t5-base,INT8,0.4220,0.5571,0.5554,0.5323,-0.001
3,flan-t5-base,INT4,0.4033,0.5404,0.5249,0.5094,-0.024
4,flan-t5-large,FP32,0.5483,0.6842,0.6968,0.6639,0.000
5,flan-t5-large,BF16,0.5490,0.6855,0.6982,0.6650,0.001
6,flan-t5-large,INT8,0.5490,0.6848,0.6984,0.6647,0.001
7,flan-t5-large,INT4,0.5303,0.6687,0.6806,0.6476,-0.016
8,flan-t5-base-finetuned,FP32,0.4440,0.5674,0.5840,0.5501,0.000
9,flan-t5-base-finetuned,BF16,0.4423,0.5661,0.5838,0.5490,-0.001
